## CelebA OOD Analysis (Per Attribute)

This notebook analyzes OOD certification results under `certify_ood`.

For each attribute folder, it creates:
- Separate result tables
- The same core curves (Certified Accuracy, Mean Radius, Abstain Rate)
- Eigen-spectrum plots from `eigenvalues.npz`

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATASET = 'celeba'
BASE = repo_root / 'output' / 'smile_classification' / DATASET
# Update only this line if your folder name differs
CERTIFY_DIR = BASE / 'certify_ood'

SIGMA_VALUES = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

def sigma_tag(v: float) -> str:
    return f'sigma_{v:.2f}'.replace('.', '_')

def mode_label(mode_name: str) -> str:
    return mode_name.replace('_', ' ').title()

def load_json(path: Path):
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def to_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(['1', 'true', 't', 'yes'])

def summarize_results_csv(csv_path: Path):
    if not csv_path.exists():
        return {}
    dfr = pd.read_csv(csv_path)
    out = {}

    if 'label' in dfr.columns and 'certified_correct' in dfr.columns:
        cc = to_bool_series(dfr['certified_correct'])
        total = len(dfr)
        smile_mask = dfr['label'] == 1
        no_smile_mask = dfr['label'] == 0

        smile_total = int(smile_mask.sum())
        no_smile_total = int(no_smile_mask.sum())
        smile_correct = int((smile_mask & cc).sum())
        no_smile_correct = int((no_smile_mask & cc).sum())

        out['smile_acc'] = (smile_correct / smile_total) if smile_total > 0 else np.nan
        out['no_smile_acc'] = (no_smile_correct / no_smile_total) if no_smile_total > 0 else np.nan

    for col in ['nn_ood_count', 'mc_ood_count', 'nn_ood_frac', 'mc_ood_frac']:
        if col in dfr.columns:
            out[f'mean_{col}'] = pd.to_numeric(dfr[col], errors='coerce').mean()

    return out

print('repo_root:', repo_root)
print('CERTIFY_DIR:', CERTIFY_DIR)
print('CERTIFY_DIR exists:', CERTIFY_DIR.exists())

## Build Per-Run DataFrame (Attribute / Mode / Sigma)

Expected layout:
- `certify_ood/<attribute>/<mode>/sigma_x_xx/metrics.json`
- Optional: `results.csv`, `eigenvalues.npz`

In [ ]:
rows = []

if not CERTIFY_DIR.exists():
    print('certify_ood directory not found. Update CERTIFY_DIR first.')
else:
    attr_dirs = sorted([p for p in CERTIFY_DIR.iterdir() if p.is_dir()])
    print(f'Found {len(attr_dirs)} attribute folders')

    for attr_dir in attr_dirs:
        attribute = attr_dir.name
        mode_dirs = sorted([p for p in attr_dir.iterdir() if p.is_dir()])

        for mode_dir in mode_dirs:
            mode_raw = mode_dir.name
            parts = mode_raw.split('_')
            space = parts[0].title() if len(parts) > 0 else 'Unknown'
            smoothing = parts[1].title() if len(parts) > 1 else 'Unknown'

            for s in SIGMA_VALUES:
                s_tag = sigma_tag(s)
                run_dir = mode_dir / s_tag
                if not run_dir.exists():
                    continue

                metrics = load_json(run_dir / 'metrics.json')
                if metrics is None:
                    continue

                vol = metrics.get('volume', {}) if isinstance(metrics, dict) else {}
                geo = vol.get('geometry', {}) if isinstance(vol, dict) else {}

                csv_stats = summarize_results_csv(run_dir / 'results.csv')

                row = {
                    'attribute': attribute,
                    'mode_raw': mode_raw,
                    'mode': mode_label(mode_raw),
                    'space': space,
                    'smoothing': smoothing,
                    'sigma': float(s),
                    'certified_acc': metrics.get('certified_accuracy', np.nan),
                    'mean_radius': metrics.get('mean_radius', np.nan),
                    'abstain_rate': metrics.get('abstain_rate', np.nan),
                    'total': metrics.get('total_test_samples', np.nan),
                    'certified_correct': metrics.get('certified_correct', np.nan),
                    'smile_acc': csv_stats.get('smile_acc', metrics.get('class_smile_accuracy', np.nan)),
                    'no_smile_acc': csv_stats.get('no_smile_acc', metrics.get('class_no_smile_accuracy', np.nan)),
                    'log_v_iso_geo': geo.get('log_v_iso_geo', np.nan),
                    'mean_log_v_mani_geo': geo.get('mean_log_v_mani_geo_max_norm', geo.get('mean_log_v_mani_geo', np.nan)),
                    'nn_ood_count': csv_stats.get('mean_nn_ood_count', np.nan),
                    'mc_ood_count': csv_stats.get('mean_mc_ood_count', np.nan),
                    'nn_ood_frac': csv_stats.get('mean_nn_ood_frac', np.nan),
                    'mc_ood_frac': csv_stats.get('mean_mc_ood_frac', np.nan),
                    'run_dir': str(run_dir),
                }
                rows.append(row)

if not rows:
    print('No completed OOD runs found.')
    df_ood = pd.DataFrame()
else:
    df_ood = pd.DataFrame(rows).sort_values(['attribute', 'space', 'smoothing', 'sigma'])
    print('Rows loaded:', len(df_ood))
    print('Attributes:', sorted(df_ood['attribute'].unique().tolist()))

df_ood.head(10)

## Separate Tables for Each Attribute

In [ ]:
if 'df_ood' not in globals() or df_ood.empty:
    print('No data loaded yet.')
else:
    table_cols = [
        'mode', 'sigma', 'certified_acc', 'mean_radius', 'abstain_rate',
        'total', 'certified_correct', 'smile_acc', 'no_smile_acc',
        'log_v_iso_geo', 'mean_log_v_mani_geo',
        'nn_ood_count', 'mc_ood_count', 'nn_ood_frac', 'mc_ood_frac'
    ]

    for attr in sorted(df_ood['attribute'].unique()):
        print('')
        print('=' * 110)
        print(f'ATTRIBUTE: {attr}')
        print('=' * 110)
        sub = df_ood[df_ood['attribute'] == attr].copy()
        sub = sub.sort_values(['space', 'smoothing', 'sigma'])
        display(sub[table_cols].round(4))

## Same Graphs Per Attribute

For each attribute, plot:
- Certified Accuracy vs sigma
- Mean Radius vs sigma
- Abstain Rate vs sigma

In [ ]:
if 'df_ood' not in globals() or df_ood.empty:
    print('No data loaded yet.')
else:
    for attr in sorted(df_ood['attribute'].unique()):
        sub_attr = df_ood[df_ood['attribute'] == attr].copy()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        mode_names = sorted(sub_attr['mode'].unique())
        for mode_name in mode_names:
            sub = sub_attr[sub_attr['mode'] == mode_name].sort_values('sigma')
            if sub.empty:
                continue
            axes[0].plot(sub['sigma'], sub['certified_acc'], 'o-', linewidth=2, label=mode_name)
            axes[1].plot(sub['sigma'], sub['mean_radius'], 'o-', linewidth=2, label=mode_name)
            axes[2].plot(sub['sigma'], sub['abstain_rate'], 'o-', linewidth=2, label=mode_name)

        axes[0].set_title('Certified Accuracy vs sigma')
        axes[1].set_title('Mean Radius vs sigma')
        axes[2].set_title('Abstain Rate vs sigma')

        axes[0].set_ylabel('Certified Accuracy')
        axes[1].set_ylabel('Mean Radius')
        axes[2].set_ylabel('Abstain Rate')

        for ax in axes:
            ax.set_xlabel('sigma')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)

        plt.suptitle(f'CelebA OOD Results - {attr}', fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

## Eigen Graph Per Attribute

Loads `eigenvalues.npz` for each attribute/mode/sigma and plots mean normalized spectrum.

In [ ]:
if 'df_ood' not in globals() or df_ood.empty:
    print('No data loaded yet.')
else:
    for attr in sorted(df_ood['attribute'].unique()):
        sub_attr = df_ood[df_ood['attribute'] == attr].copy()
        spectra = []

        for _, r in sub_attr.iterrows():
            run_dir = Path(r['run_dir'])
            ep = run_dir / 'eigenvalues.npz'
            if not ep.exists():
                continue

            try:
                npz = np.load(ep)
                if 'eigenvalues_norm_max' in npz.files:
                    arr = np.asarray(npz['eigenvalues_norm_max'], dtype=np.float64)
                elif 'eigenvalues' in npz.files:
                    eig = np.asarray(npz['eigenvalues'], dtype=np.float64)
                    m = np.nanmax(eig, axis=1, keepdims=True)
                    m[m <= 0] = 1.0
                    arr = eig / m
                else:
                    continue

                mean_spec = np.nanmean(arr, axis=0)
                spectra.append((r['mode'], float(r['sigma']), mean_spec))
            except Exception as e:
                print(f'Skip eigen file {ep}: {e}')

        if not spectra:
            print(f'No eigen spectra found for attribute: {attr}')
            continue

        plt.figure(figsize=(10, 5))
        for mode_name, sig, spec in spectra:
            x = np.arange(1, len(spec) + 1)
            plt.plot(x, spec, linewidth=1.8, label=f'{mode_name} @ sigma={sig:.2f}')

        plt.title(f'Eigen Spectrum (normalized) - {attr}')
        plt.xlabel('PCA component')
        plt.ylabel('mean normalized eigenvalue')
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=7)
        plt.tight_layout()
        plt.show()

## Per-Sample NN/MC OOD Counts (from results.csv)

This section is for sample-level reporting.

Use it to present, per sample:
- `nn_ood_count`, `nn_ood_frac`
- `mc_ood_count`, `mc_ood_frac`
- plus `correct` and `certified_correct`

The main table keeps means across samples; this section shows the raw counts for a chosen attribute/mode/sigma.

In [ ]:
# Pick one run to inspect sample-level counts
ATTRIBUTE_VIEW = 'wearing_hat'   # folder name under certify_ood
MODE_VIEW = 'pixel_isotropic'    # e.g. pixel_isotropic, pixel_manifold, latent_isotropic, latent_manifold
SIGMA_VIEW = 0.15

run_dir = CERTIFY_DIR / ATTRIBUTE_VIEW / MODE_VIEW / sigma_tag(SIGMA_VIEW)
csv_path = run_dir / 'results.csv'

print('Run dir:', run_dir)
print('results.csv exists:', csv_path.exists())

if not csv_path.exists():
    print('No results.csv for this selection. Change ATTRIBUTE_VIEW / MODE_VIEW / SIGMA_VIEW.')
else:
    dfr = pd.read_csv(csv_path)

    keep_cols = [
        'idx', 'image_path', 'label', 'pred', 'radius',
        'correct', 'certified_correct',
        'nn_ood_count', 'nn_ood_frac', 'mc_ood_count', 'mc_ood_frac'
    ]
    keep_cols = [c for c in keep_cols if c in dfr.columns]

    # Main per-sample table for reporting
    display(dfr[keep_cols].head(50))

    # Optional: full table sorted by OOD counts
    if {'nn_ood_count', 'mc_ood_count'}.issubset(dfr.columns):
        dfr_sorted = dfr.sort_values(['mc_ood_count', 'nn_ood_count'], ascending=False)
        print('\nTop samples by MC/NN OOD count:')
        display(dfr_sorted[keep_cols].head(30))

    # Summary numbers (to compare with means in df_ood)
    print('\nSummary from this results.csv')
    for col in ['nn_ood_count', 'nn_ood_frac', 'mc_ood_count', 'mc_ood_frac']:
        if col in dfr.columns:
            vals = pd.to_numeric(dfr[col], errors='coerce')
            print(f"{col}: mean={vals.mean():.4f}, median={vals.median():.4f}, min={vals.min():.4f}, max={vals.max():.4f}")

    # Distribution plots for counts
    plot_cols = [c for c in ['nn_ood_count', 'mc_ood_count'] if c in dfr.columns]
    if plot_cols:
        fig, axes = plt.subplots(1, len(plot_cols), figsize=(6 * len(plot_cols), 4))
        if len(plot_cols) == 1:
            axes = [axes]
        for ax, c in zip(axes, plot_cols):
            vals = pd.to_numeric(dfr[c], errors='coerce').dropna()
            ax.hist(vals, bins=30)
            ax.set_title(f'{c} distribution')
            ax.set_xlabel(c)
            ax.set_ylabel('count')
            ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## Distribution Dashboard (Presentation Friendly)

This dashboard gives a cleaner visual summary for one selected run:
- Histogram + mean/median lines
- Empirical CDF (how quickly counts accumulate)
- Compact box-style comparison for NN vs MC counts
- Optional normalized fractions (`nn_ood_frac`, `mc_ood_frac`)


In [ ]:
# Presentation-friendly distribution plots for one selected run
# Reuses ATTRIBUTE_VIEW / MODE_VIEW / SIGMA_VIEW from the previous cell.

run_dir = CERTIFY_DIR / ATTRIBUTE_VIEW / MODE_VIEW / sigma_tag(SIGMA_VIEW)
csv_path = run_dir / 'results.csv'

if not csv_path.exists():
    print('No results.csv for this selection. Change ATTRIBUTE_VIEW / MODE_VIEW / SIGMA_VIEW.')
else:
    dfr = pd.read_csv(csv_path)

    # ---- helpers ----
    def _num(s):
        return pd.to_numeric(s, errors='coerce').dropna().to_numpy()

    def _ecdf(x):
        xs = np.sort(x)
        ys = np.arange(1, len(xs) + 1) / len(xs)
        return xs, ys

    # ---- count distributions ----
    count_cols = [c for c in ['nn_ood_count', 'mc_ood_count'] if c in dfr.columns]

    if count_cols:
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

        colors = {'nn_ood_count': '#1f77b4', 'mc_ood_count': '#d62728'}

        # (1) Histograms with mean/median markers
        for c in count_cols:
            vals = _num(dfr[c])
            if len(vals) == 0:
                continue
            axes[0].hist(vals, bins=min(30, max(8, int(np.sqrt(len(vals))))), alpha=0.45,
                         label=f"{c} (n={len(vals)})", color=colors.get(c, None), edgecolor='white')
            axes[0].axvline(vals.mean(), linestyle='--', linewidth=2,
                            color=colors.get(c, 'black'), alpha=0.9,
                            label=f"{c} mean={vals.mean():.2f}")
            axes[0].axvline(np.median(vals), linestyle=':', linewidth=2,
                            color=colors.get(c, 'black'), alpha=0.9,
                            label=f"{c} median={np.median(vals):.2f}")

        axes[0].set_title('Count Distribution')
        axes[0].set_xlabel('OOD count')
        axes[0].set_ylabel('Number of samples')
        axes[0].grid(True, alpha=0.25)
        axes[0].legend(fontsize=8)

        # (2) ECDFs
        for c in count_cols:
            vals = _num(dfr[c])
            if len(vals) == 0:
                continue
            xs, ys = _ecdf(vals)
            axes[1].plot(xs, ys, linewidth=2.2, label=c, color=colors.get(c, None))

        axes[1].set_title('Empirical CDF')
        axes[1].set_xlabel('OOD count')
        axes[1].set_ylabel('Fraction of samples <= x')
        axes[1].set_ylim(0, 1.02)
        axes[1].grid(True, alpha=0.25)
        axes[1].legend(fontsize=9)

        # (3) Box comparison
        box_data, box_labels = [], []
        for c in count_cols:
            vals = _num(dfr[c])
            if len(vals) == 0:
                continue
            box_data.append(vals)
            box_labels.append(c)

        if box_data:
            b = axes[2].boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.55)
            for patch, lbl in zip(b['boxes'], box_labels):
                patch.set_facecolor(colors.get(lbl, '#bbbbbb'))
                patch.set_alpha(0.45)
            for med in b['medians']:
                med.set_color('black')
                med.set_linewidth(2)

        axes[2].set_title('Spread Comparison (Boxplot)')
        axes[2].set_ylabel('OOD count')
        axes[2].grid(True, alpha=0.25)

        plt.suptitle(
            f"OOD Count Dashboard | attr={ATTRIBUTE_VIEW}, mode={MODE_VIEW}, sigma={SIGMA_VIEW:.2f}",
            fontsize=12,
            y=1.03,
        )
        plt.tight_layout()
        plt.show()

    # ---- optional fraction distributions ----
    frac_cols = [c for c in ['nn_ood_frac', 'mc_ood_frac'] if c in dfr.columns]
    if frac_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
        colors_frac = {'nn_ood_frac': '#2ca02c', 'mc_ood_frac': '#9467bd'}

        for c in frac_cols:
            vals = _num(dfr[c])
            if len(vals) == 0:
                continue
            axes[0].hist(vals, bins=25, alpha=0.5, label=f"{c} (mean={vals.mean():.3f})",
                         color=colors_frac.get(c, None), edgecolor='white')
            xs, ys = _ecdf(vals)
            axes[1].plot(xs, ys, linewidth=2.2, label=c, color=colors_frac.get(c, None))

        axes[0].set_title('Fraction Distribution')
        axes[0].set_xlabel('OOD fraction')
        axes[0].set_ylabel('Number of samples')
        axes[0].grid(True, alpha=0.25)
        axes[0].legend(fontsize=8)

        axes[1].set_title('Fraction ECDF')
        axes[1].set_xlabel('OOD fraction')
        axes[1].set_ylabel('Fraction of samples <= x')
        axes[1].set_ylim(0, 1.02)
        axes[1].grid(True, alpha=0.25)
        axes[1].legend(fontsize=9)

        plt.suptitle(
            f"OOD Fraction Dashboard | attr={ATTRIBUTE_VIEW}, mode={MODE_VIEW}, sigma={SIGMA_VIEW:.2f}",
            fontsize=12,
            y=1.03,
        )
        plt.tight_layout()
        plt.show()
